# C5 — MCP Library Dashboard
Upload your existing project ZIP, run the cells, then open the dashboard. Runs on Colab, not on your computer.

In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
import io, shutil

uploaded = files.upload()
zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if not zip_name:
    raise ValueError('Upload C5_Library_MCP.zip.')

ROOT = Path('/content/C5_Library_MCP')
ROOT.mkdir(parents=True, exist_ok=True)
skipped = {'.env', '__pycache__', '.git', '.venv', 'venv', '__MACOSX'}
with ZipFile(io.BytesIO(uploaded[zip_name])) as archive:
    for member in archive.infolist():
        parts = PurePosixPath(member.filename).parts
        if not parts or member.is_dir():
            continue
        if parts[0] == 'C5_Library_MCP':
            parts = parts[1:]
        if not parts or any(part in skipped or part in {'.', '..'} for part in parts):
            continue
        if parts[-1].endswith(('.pyc', '.pyo')):
            continue
        target = ROOT.joinpath(*parts).resolve()
        if ROOT.resolve() not in target.parents:
            raise ValueError('Unsafe ZIP path: ' + member.filename)
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as src, target.open('wb') as dst:
            shutil.copyfileobj(src, dst)

# Do not load API keys that might have been included in the ZIP.
(ROOT / '.env').unlink(missing_ok=True)
Path(zip_name).unlink(missing_ok=True)  # Remove the uploaded archive from Colab storage.
uploaded.clear()
for name in ('dashboard.py', 'dashboard.html', 'server.py', 'agent.py', 'library.py', 'requirements.txt'):
    if not (ROOT / name).is_file():
        raise FileNotFoundError(f'{name} missing from ZIP')
print('Project ready. Dashboard and three MCP tools preserved.')

In [ ]:
%pip -q install -r /content/C5_Library_MCP/requirements.txt

In [ ]:
import os
from getpass import getpass

key = os.environ.get('GEMINI_API_KEY', '').strip()
if not key:
    try:
        from google.colab import userdata
        key = (userdata.get('GEMINI_API_KEY') or '').strip()
    except Exception:
        pass
if not key:
    key = getpass('Gemini API key: ').strip()
if not key:
    raise ValueError('Gemini API key is required for chatbot requests.')
os.environ['GEMINI_API_KEY'] = key
os.environ.setdefault('GEMINI_MODEL', 'gemini-3.5-flash-lite')
print('Gemini configured.')

In [ ]:
import subprocess, sys

check = subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
    cwd=ROOT, capture_output=True, text=True
)
print(check.stderr[-1800:] if check.stderr else check.stdout[-1800:])
if check.returncode != 0:
    raise RuntimeError('Unit tests failed')

mcp_check = subprocess.run(
    [sys.executable, 'check_mcp.py'], cwd=ROOT,
    capture_output=True, text=True, timeout=90
)
print('MCP:', next((line for line in mcp_check.stdout.splitlines() if 'Discovered tools:' in line), 'Check completed'))
if mcp_check.returncode != 0:
    print(mcp_check.stderr[-2000:])
    raise RuntimeError('MCP checks failed')

### Optional: reset or terminal fallback (set a switch to `True`)

In [ ]:
RESET_BOOKS = False
RUN_TERMINAL_AGENT = False

if RESET_BOOKS:
    subprocess.run([sys.executable, 'reset_books.py'], cwd=ROOT, check=True)
    print('Reservations cleared. Refresh the dashboard.')

if RUN_TERMINAL_AGENT:
    request = 'Search for Python books. Do not reserve anything.'
    result = subprocess.run(
        [sys.executable, 'agent.py', '--once', request],
        cwd=ROOT, env=os.environ.copy(), capture_output=True, text=True, timeout=180
    )
    print(result.stdout[-5000:])
    if result.returncode:
        print(result.stderr[-2500:])

### Dashboard

In [ ]:
import subprocess, sys, time, urllib.request, html
from IPython.display import HTML, display
from google.colab import output

PORT = 8000
if 'dashboard_process' not in globals() or dashboard_process.poll() is not None:
    log_handle = open('/content/mcp_dashboard.log', 'a', encoding='utf-8')
    dashboard_process = subprocess.Popen(
        [sys.executable, '-u', str(ROOT / 'dashboard.py'), str(PORT)],
        cwd=str(ROOT), env=os.environ.copy(),
        stdout=log_handle, stderr=subprocess.STDOUT,
        start_new_session=True
    )

for attempt in range(40):
    if dashboard_process.poll() is not None:
        raise RuntimeError('Dashboard exited; read /content/mcp_dashboard.log')
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/api/books', timeout=2) as response:
            if response.status == 200:
                break
    except Exception:
        time.sleep(0.25)
else:
    raise RuntimeError('Dashboard did not start. Read /content/mcp_dashboard.log')

print('Dashboard running.')
try:
    from google.colab.output import eval_js
    proxy_url = eval_js(f'google.colab.kernel.proxyPort({PORT})')
    if not isinstance(proxy_url, str) or not proxy_url.startswith('https://'):
        raise ValueError('Colab URL unavailable')
    display(HTML('<a target="_blank" rel="noopener noreferrer" '
        'style="display:inline-block;padding:12px 20px;background:#1677e7;'
        'color:white;border-radius:8px;text-decoration:none;font-weight:600" '
        f'href="{html.escape(proxy_url, quote=True)}">Open Library Dashboard</a>'))
except Exception as exc:
    print('Use the inline dashboard in the next cell:', type(exc).__name__)

If the button does not open, run this cell to use the dashboard inside Colab.

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(PORT, height='900')